In [4]:
import sympy as sp
# 从模块中正确导入需要的类和函数
from dual_deriver import DualDeriver, SUM, TERM, render_dual

# 1. 初始化推导器 (默认求解 min 问题)
deriver = DualDeriver(sense="min")

# 2. 定义纯数学符号 (用于作为下标索引)
i = sp.symbols("i")

# 3. 定义已知参数 (Parameters)
c = sp.IndexedBase("c")
b = sp.IndexedBase("b")

# 4. 声明决策变量 (Variables)
# 语法: deriver.declare_var(变量名, 集合名称列表, 对应的符号列表, 下界)
x = deriver.declare_var("x", ["I"], free_symbols=[i], lower=0)

# 5. 设置目标函数: min \sum_{i \in I} c_i x_i
deriver.set_objective(
    SUM(c[i] * x[i], (i, "I"))
)

# 6. 添加约束: x_i >= b_i, \forall i \in I
deriver.add_constraint(
    lhs = TERM(x[i]), 
    sense = ">=", 
    rhs = TERM(b[i]),
    forall = [i],            # 约束包含的循环指标
    forall_sets = ["I"],     # 循环指标对应的集合
    multiplier_name = "lambda", # 自动生成 \lambda_i 作为对偶乘子
    name = "demand_constraint"
)

# 7. 执行推导并打印 LaTeX
result = deriver.derive_dual()

print("====== dual LaTeX ======")
print(render_dual(result))

====== dual LaTeX ======
\textbf{Maximize}\quad \sum_{i \in I} {b}_{i} {\lambda}_{i}
\text{subject to:}
    {\lambda}_{i} \leq {c}_{i},\ \ \forall\, i \in I
    {\lambda}_{i} \geq 0,\ \ \forall\, i \in I


In [5]:
import sympy as sp
from dual_deriver import DualDeriver, SUM, TERM, render_dual

deriver = DualDeriver(sense="min")

x_1 = deriver.declare_var("x_1", lower=None)
x_2 = deriver.declare_var("x_2", lower=None)

deriver.set_objective(
    TERM((x_1**2) + 2*(x_2**2) + 4*x_1 - 8*x_2)
)

deriver.add_constraint(
    lhs = TERM(x_1 + x_2),
    sense = ">=",
    rhs = TERM(1),
    multiplier_name = "lambda",
    name = "constraint1"
)

deriver.add_constraint(
    lhs = TERM(x_1),
    sense = ">=",
    rhs = TERM(0),
    multiplier_name = "mu_1",
    name = "constraint2"
)

deriver.add_constraint(
    lhs = TERM(x_2),
    sense = ">=",
    rhs = TERM(0),
    multiplier_name = "mu_2",
    name = "constraint3"
)

result = deriver.derive_dual()
print("====== dual LaTeX ======")
print(render_dual(result))

====== dual LaTeX ======
\textbf{Maximize}\quad - \left(\frac{3 \lambda^{2}}{8} + \frac{\lambda \mu_{1}}{2} + \frac{\lambda \mu_{2}}{4} - \lambda + \frac{\mu_{1}^{2}}{4} - 2 \mu_{1} + \frac{\mu_{2}^{2}}{8} + 2 \mu_{2} + 12\right)
\text{subject to:}
    \lambda \geq 0
    \mu_{1} \geq 0
    \mu_{2} \geq 0
\text{(eliminated by KKT stationarity:)}
    x_{1} = \frac{\lambda}{2} + \frac{\mu_{1}}{2} - 2
    x_{2} = \frac{\lambda}{4} + \frac{\mu_{2}}{4} + 2
